# AdvectNet test on MOSDAC INSAT-3DR TIR1

This notebook downloads INSAT-3DR Imager Level-1C sector data from MOSDAC using `mdapi.py`, extracts the thermal infrared channel `IMG_TIR1`, converts counts to brightness temperature using `IMG_TIR1_TEMP`, normalizes to the same range used in the Himawari training notebook, and runs the already-trained AdvectNet model.

This notebook does **not train**. It needs a trained checkpoint from the Himawari notebook, e.g.

```python
torch.save({"model": unet.state_dict(), "bt_min": BT_MIN, "bt_max": BT_MAX, "patch": PATCH},
           "advectnet_unet_himawari_physics.pth")
```

Cadence logic:

- If MOSDAC provides `t, t+10, t+20, t+30`, the notebook performs true `+10/+20` evaluation.
- If only half-hourly INSAT frames are available, true `+10/+20` ground truth does not exist. The notebook still generates direct `+10/+20` predictions and also runs a proxy evaluation over `(t, t+30, t+60, t+90)` windows.

## 1. Setup

In [ ]:
!pip install -q requests tqdm h5py scikit-image imageio

import os, re, io, json, shutil, subprocess, datetime as dt
from pathlib import Path

import h5py
import imageio.v2 as imageio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim_fn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

## 2. Configuration

In [ ]:
# MOSDAC product for INSAT-3DR Imager L1C sector product.
DATASET_ID = "3RIMG_L1C_SGP"

# Use either latest COUNT products, or a specific date/time range.
# MOSDAC API accepted empty start/end + count in the reference notebook.
START_TIME = ""          # e.g. "2026-06-25"
END_TIME = ""            # e.g. "2026-06-25"
COUNT = "10"             # latest N files when START_TIME/END_TIME are empty

# Trained Himawari checkpoint. Auto-discovery searches /kaggle/input.
WEIGHTS_PATH = None

# MOSDAC credentials: recommended Kaggle secrets named MOSDAC_USERNAME and MOSDAC_PASSWORD.
# Do not hardcode credentials in a public notebook version.
MOSDAC_USERNAME = None
MOSDAC_PASSWORD = None

# Paths.
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
WORK = Path("/kaggle/working")
DOWNLOAD_DIR = WORK / "mosdac_tir1_downloads"
OUT_DIR = WORK / "insat3dr_tir1_advectnet_outputs"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Model/test settings.
PATCH = 256
STRIDE = 256
BATCH = 8
BT_MIN, BT_MAX = 180.0, 310.0   # must match Himawari notebook normalization
MIN_VALID = 0.25
MAX_EVAL_WINDOWS = None         # set 3 for quick smoke test
SAVE_FULL_NPZ = False

print("DATASET_ID:", DATASET_ID)
print("DOWNLOAD_DIR:", DOWNLOAD_DIR)
print("OUT_DIR:", OUT_DIR)

## 3. Locate MOSDAC API files and checkpoint

In [ ]:
def all_files(roots=ROOTS):
    hits = []
    for root in roots:
        if root.exists():
            hits.extend([p for p in root.rglob("*") if p.is_file()])
    return sorted(set(hits), key=lambda p: str(p).lower())

def locate_helpers(extra_roots=None):
    roots = list(ROOTS)
    if extra_roots:
        roots.extend(Path(x) for x in extra_roots)
    files = all_files(roots)
    mdapi_hits = [p for p in files if p.name == "mdapi.py"]
    config_hits = [p for p in files if p.name == "config.json"]
    weight_hits = [p for p in files if p.suffix.lower() in [".pth", ".pt", ".ckpt"]]
    return files, mdapi_hits, config_hits, weight_hits

files, mdapi_hits, config_hits, weight_hits = locate_helpers()

# Fallback for the exact helper dataset used in the MOSDAC reference notebook.
# This works only if the Kaggle dataset is public/accessible to the session.
if not (mdapi_hits and config_hits):
    try:
        import kagglehub
        mosdac_path = kagglehub.dataset_download("krityapriyabhaumik/mosdac")
        print("downloaded/found MOSDAC helper dataset via kagglehub:", mosdac_path)
        files, mdapi_hits, config_hits, weight_hits = locate_helpers([mosdac_path])
    except Exception as e:
        print("kagglehub fallback failed:", repr(e))

print("mdapi candidates:")
for p in mdapi_hits[:20]: print(" ", p)
print("config candidates:")
for p in config_hits[:20]: print(" ", p)
print("checkpoint candidates:")
for p in weight_hits[:20]: print(" ", p)

if not mdapi_hits or not config_hits:
    print("\nFiles visible under /kaggle/input:")
    for p in [x for x in files if str(x).startswith("/kaggle/input")][:120]:
        print(" ", p)
    raise FileNotFoundError(
        "MOSDAC helper files are missing. In Kaggle, click Add Data and attach the dataset "
        "krityapriyabhaumik/mosdac, which must contain mdapi.py and config.json. "
        "Your current session appears to have only the model/data-isro input mounted."
    )

shutil.copy2(mdapi_hits[0], WORK / "mdapi.py")
shutil.copy2(config_hits[0], WORK / "config.json")
print("copied mdapi.py and config.json to", WORK)

if WEIGHTS_PATH is None:
    preferred = [p for p in weight_hits if re.search(r"advect|hima|unet|physics", p.name, re.I)]
    WEIGHTS_PATH = str((preferred or weight_hits)[0]) if weight_hits else None
print("WEIGHTS_PATH:", WEIGHTS_PATH)
assert WEIGHTS_PATH and Path(WEIGHTS_PATH).exists(), "trained Himawari checkpoint not found. Upload the .pth/.pt file as Kaggle input."

## 4. Credentials and MOSDAC download

In [ ]:
# Read Kaggle secrets if available.
if MOSDAC_USERNAME is None or MOSDAC_PASSWORD is None:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        MOSDAC_USERNAME = MOSDAC_USERNAME or secrets.get_secret("MOSDAC_USERNAME")
        MOSDAC_PASSWORD = MOSDAC_PASSWORD or secrets.get_secret("MOSDAC_PASSWORD")
    except Exception as e:
        print("Kaggle secrets unavailable or not set:", repr(e))

assert MOSDAC_USERNAME and MOSDAC_PASSWORD, (
    "Set MOSDAC_USERNAME and MOSDAC_PASSWORD above, or create Kaggle secrets with those names."
)

cfg_path = WORK / "config.json"
with open(cfg_path, "r") as f:
    cfg = json.load(f)

cfg["user_credentials"]["username/email"] = MOSDAC_USERNAME
cfg["user_credentials"]["password"] = MOSDAC_PASSWORD
cfg["search_parameters"]["datasetId"] = DATASET_ID
cfg["search_parameters"]["startTime"] = START_TIME
cfg["search_parameters"]["endTime"] = END_TIME
cfg["search_parameters"]["count"] = COUNT
cfg["search_parameters"]["boundingBox"] = cfg["search_parameters"].get("boundingBox", "")
cfg["search_parameters"]["gId"] = cfg["search_parameters"].get("gId", "")
cfg["download_settings"]["download_path"] = str(DOWNLOAD_DIR)
cfg["download_settings"]["skip_user_input"] = True
cfg["download_settings"]["organize_by_date"] = False

with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=4)

print("MOSDAC config prepared. Credential values are not printed.")
print(json.dumps(cfg["search_parameters"], indent=2))

res = subprocess.run(["python", str(WORK / "mdapi.py")], cwd=str(WORK), text=True, capture_output=True)

def redact_log(s):
    s = s or ""
    for secret in [MOSDAC_USERNAME, MOSDAC_PASSWORD]:
        if secret:
            s = s.replace(secret, "<redacted>")
    return s

print(redact_log(res.stdout[-4000:]))
if res.returncode != 0:
    print(redact_log(res.stderr[-4000:]))

# Scrub credentials from the working config so they are not preserved in notebook outputs.
with open(cfg_path, "r") as f:
    scrub_cfg = json.load(f)
scrub_cfg["user_credentials"]["username/email"] = "<redacted>"
scrub_cfg["user_credentials"]["password"] = "<redacted>"
with open(cfg_path, "w") as f:
    json.dump(scrub_cfg, f, indent=4)

assert res.returncode == 0, "mdapi.py failed. Check datasetId/date/count/credentials above."

downloaded = sorted(DOWNLOAD_DIR.rglob("*.h5"))
print("downloaded h5 files:", len(downloaded))
for p in downloaded[:20]: print(" ", p.name, p.stat().st_size/1e6, "MB")
assert downloaded, "No HDF5 files downloaded."

## 5. Inspect one HDF5 file

In [ ]:
sample = downloaded[0]
print("sample:", sample)
with h5py.File(sample, "r") as f:
    print("root attrs:")
    for k, v in list(f.attrs.items())[:40]:
        vv = v.decode("utf-8", errors="ignore") if isinstance(v, bytes) else v
        print(" ", k, repr(vv)[:160])
    print("\ndatasets:")
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, "shape=", obj.shape, "dtype=", obj.dtype)
    f.visititems(visit)
    assert "IMG_TIR1" in f, "Expected dataset IMG_TIR1 not found. See printed dataset list."
    assert "IMG_TIR1_TEMP" in f, "Expected calibration LUT IMG_TIR1_TEMP not found. See printed dataset list."

## 6. Read TIR1 as normalized brightness temperature

In [ ]:
def bstr(x):
    if isinstance(x, bytes):
        return x.decode("utf-8", errors="ignore")
    if hasattr(x, "tolist"):
        x = x.tolist()
        if isinstance(x, bytes):
            return x.decode("utf-8", errors="ignore")
    return x

def parse_time_from_name(path):
    name = Path(path).name
    m = re.search(r"3RIMG_(\d{2}[A-Z]{3}\d{4})_(\d{4})_", name)
    if not m:
        raise ValueError(f"cannot parse INSAT-3DR timestamp from {name}")
    return dt.datetime.strptime(m.group(1) + m.group(2), "%d%b%Y%H%M")

def read_tir1_bt(path):
    with h5py.File(path, "r") as f:
        raw = np.asarray(f["IMG_TIR1"])
        if raw.ndim == 3:
            raw = raw[0]
        raw = raw.astype(np.int64)

        lut = np.asarray(f["IMG_TIR1_TEMP"]).astype(np.float32).reshape(-1)
        fill = f["IMG_TIR1"].attrs.get("_FillValue", None)
        if hasattr(fill, "tolist"):
            fill = fill.tolist()
        if isinstance(fill, (list, tuple, np.ndarray)):
            fill = fill[0]

        valid = (raw >= 0) & (raw < len(lut))
        if fill is not None:
            valid &= (raw != int(fill))
        bt = lut[np.clip(raw, 0, len(lut)-1)].astype(np.float32)
        valid &= np.isfinite(bt) & (bt > 100) & (bt < 400)

        meta = {k: bstr(v) for k, v in f.attrs.items()}

    norm = (np.clip(bt, BT_MIN, BT_MAX) - BT_MIN) / (BT_MAX - BT_MIN)
    norm = np.where(valid, norm, 0.0).astype(np.float32)
    return norm, valid.astype(bool), bt.astype(np.float32), meta

rows = []
for p in downloaded:
    try:
        rows.append((parse_time_from_name(p), p))
    except Exception as e:
        print("skip unparsable", p.name, e)
rows.sort()
assert len(rows) >= 2, "Need at least two time frames. Increase COUNT."

frames, masks, bts, times, metas = [], [], [], [], []
for t, p in rows:
    norm, valid, bt, meta = read_tir1_bt(p)
    frames.append(norm)
    masks.append(valid)
    bts.append(bt)
    times.append(t)
    metas.append(meta)
    vals = bt[valid]
    print(f"{t:%Y-%m-%d %H:%M} shape={norm.shape} valid={valid.mean():.3f} BT[min/mean/max]={vals.min():.1f}/{vals.mean():.1f}/{vals.max():.1f}")

# Crop to common shape if products differ by a few rows/cols.
hmin = min(x.shape[0] for x in frames); wmin = min(x.shape[1] for x in frames)
frames = np.stack([x[:hmin, :wmin] for x in frames])
masks = np.stack([x[:hmin, :wmin] for x in masks])
bts = np.stack([x[:hmin, :wmin] for x in bts])
N, H, W = frames.shape
print("sequence:", frames.shape, "from", times[0], "to", times[-1])
print("time deltas:", sorted(set((times[i+1]-times[i]).total_seconds()/60 for i in range(N-1))))

## 7. Load trained AdvectNet checkpoint

In [ ]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

raft = raft_small(weights=Raft_Small_Weights.DEFAULT).to(dev).eval()
for p in raft.parameters():
    p.requires_grad_(False)

@torch.no_grad()
def raft_flow(a, b):
    a3 = (a * 2 - 1).repeat(1, 3, 1, 1)
    b3 = (b * 2 - 1).repeat(1, 3, 1, 1)
    return raft(a3, b3)[-1]

def backwarp(img, flow):
    B, C, h, w = img.shape
    yy, xx = torch.meshgrid(torch.arange(h, device=img.device), torch.arange(w, device=img.device), indexing="ij")
    grid = torch.stack((xx, yy), 0).float()[None].repeat(B, 1, 1, 1) + flow
    gx = 2 * grid[:, 0] / max(w - 1, 1) - 1
    gy = 2 * grid[:, 1] / max(h - 1, 1) - 1
    return F.grid_sample(img, torch.stack((gx, gy), -1), mode="bilinear", padding_mode="border", align_corners=True)

def inter_flows(F03, F30, alpha):
    Ft0 = -(1 - alpha) * alpha * F03 + alpha * alpha * F30
    Ft3 = (1 - alpha) ** 2 * F03 - alpha * (1 - alpha) * F30
    return Ft0, Ft3

def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, 1, 1), nn.GroupNorm(8, o), nn.GELU())

class UNet(nn.Module):
    def __init__(self, ic=8, b=32):
        super().__init__()
        self.e1 = nn.Sequential(cbr(ic, b), cbr(b, b))
        self.e2 = nn.Sequential(cbr(b, 2*b), cbr(2*b, 2*b))
        self.e3 = nn.Sequential(cbr(2*b, 4*b), cbr(4*b, 4*b))
        self.pool = nn.MaxPool2d(2)
        self.d2 = nn.Sequential(cbr(4*b+2*b, 2*b), cbr(2*b, 2*b))
        self.d1 = nn.Sequential(cbr(2*b+b, b), cbr(b, b))
        self.out = nn.Conv2d(b, 2, 3, 1, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        up = lambda z: F.interpolate(z, scale_factor=2, mode="bilinear", align_corners=False)
        d2 = self.d2(torch.cat([up(e3), e2], 1)); d1 = self.d1(torch.cat([up(d2), e1], 1))
        o = self.out(d1)
        return torch.sigmoid(o[:, 0:1]), torch.tanh(o[:, 1:2])

unet = UNet().to(dev)

def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for key in ["model", "unet", "state_dict", "model_state_dict"]:
            if key in ckpt and isinstance(ckpt[key], dict):
                ckpt = ckpt[key]
                break
    out = {}
    for k, v in ckpt.items():
        nk = k
        for pref in ["module.", "unet.", "model."]:
            if nk.startswith(pref):
                nk = nk[len(pref):]
        out[nk] = v
    return out

ckpt = torch.load(WEIGHTS_PATH, map_location=dev)

# New month-scale checkpoints contain both RAFT and U-Net. Older checkpoints may contain only U-Net.
if isinstance(ckpt, dict) and "raft" in ckpt:
    r_missing, r_unexpected = raft.load_state_dict(ckpt["raft"], strict=False)
    print("raft missing:", r_missing)
    print("raft unexpected:", r_unexpected)

if isinstance(ckpt, dict) and "unet" in ckpt:
    state = extract_state_dict(ckpt["unet"])
else:
    state = extract_state_dict(ckpt)

missing, unexpected = unet.load_state_dict(state, strict=False)
print("unet missing:", missing)
print("unet unexpected:", unexpected)
assert len(missing) == 0, "Checkpoint architecture mismatch."
raft.eval()
unet.eval()
print("checkpoint loaded")

## 8. Patchwise inference utilities

In [ ]:
def starts(n, patch=PATCH, stride=STRIDE):
    if n <= patch:
        return [0]
    vals = list(range(0, n - patch + 1, stride))
    if vals[-1] != n - patch:
        vals.append(n - patch)
    return vals

def pad_min(img, patch=PATCH, value=0):
    h, w = img.shape
    hp, wp = max(h, patch), max(w, patch)
    out = np.full((hp, wp), value, dtype=img.dtype)
    out[:h, :w] = img
    return out, (h, w)

@torch.no_grad()
def infer_batch(x0, x3, alpha):
    x0t = torch.tensor(x0, dtype=torch.float32, device=dev)[:, None]
    x3t = torch.tensor(x3, dtype=torch.float32, device=dev)[:, None]
    F03 = raft_flow(x0t, x3t)
    F30 = raft_flow(x3t, x0t)
    a = torch.full((len(x0), 1, 1, 1), float(alpha), device=dev)
    Ft0, Ft3 = inter_flows(F03, F30, a)
    w0 = backwarp(x0t, Ft0)
    w3 = backwarp(x3t, Ft3)
    blend = (1 - a) * x0t + a * x3t
    ach = a.expand(-1, 1, x0t.shape[-2], x0t.shape[-1])
    mask, res = unet(torch.cat([w0, w3, Ft0, Ft3, blend, ach], 1))
    out = (mask * w0 + (1 - mask) * w3 + res).clamp(0, 1)
    return out[:, 0].detach().cpu().numpy().astype(np.float32)

def interp_tiled(f0, f3, m0, m3, alpha, patch=PATCH, stride=STRIDE, batch=BATCH, min_valid=MIN_VALID):
    f0p, (h, w) = pad_min(f0, patch, 0)
    f3p, _ = pad_min(f3, patch, 0)
    m0p, _ = pad_min(m0.astype(bool), patch, False)
    m3p, _ = pad_min(m3.astype(bool), patch, False)
    hp, wp = f0p.shape
    out = np.zeros((hp, wp), np.float32)
    wgt = np.zeros((hp, wp), np.float32)
    win1 = np.hanning(patch).astype(np.float32)
    win = np.maximum(np.outer(win1, win1), 0.05).astype(np.float32)

    jobs, coords = [], []
    for y in starts(hp, patch, stride):
        for x in starts(wp, patch, stride):
            vm = m0p[y:y+patch, x:x+patch] & m3p[y:y+patch, x:x+patch]
            if vm.mean() < min_valid:
                continue
            jobs.append((f0p[y:y+patch, x:x+patch], f3p[y:y+patch, x:x+patch]))
            coords.append((y, x))
            if len(jobs) == batch:
                pred = infer_batch(np.stack([j[0] for j in jobs]), np.stack([j[1] for j in jobs]), alpha)
                for p, (yy, xx) in zip(pred, coords):
                    out[yy:yy+patch, xx:xx+patch] += p * win
                    wgt[yy:yy+patch, xx:xx+patch] += win
                jobs, coords = [], []
    if jobs:
        pred = infer_batch(np.stack([j[0] for j in jobs]), np.stack([j[1] for j in jobs]), alpha)
        for p, (yy, xx) in zip(pred, coords):
            out[yy:yy+patch, xx:xx+patch] += p * win
            wgt[yy:yy+patch, xx:xx+patch] += win

    base = (1 - alpha) * f0p + alpha * f3p
    stitched = np.where(wgt > 0, out / np.maximum(wgt, 1e-6), base)
    return stitched[:h, :w].astype(np.float32)

print("tile grid:", len(starts(H)), "x", len(starts(W)), "=", len(starts(H)) * len(starts(W)))

## 9. Metrics

In [ ]:
def eval_metrics_single(pred, gt, valid):
    valid = valid & np.isfinite(pred) & np.isfinite(gt)
    if valid.sum() == 0:
        return dict(mse=np.nan, psnr=np.nan, ssim=np.nan)
    p = np.clip(pred[valid], 0, 1)
    g = np.clip(gt[valid], 0, 1)
    mse = float(np.mean((p - g) ** 2))
    psnr = float(10 * np.log10(1.0 / max(mse, 1e-12)))

    # SSIM needs rectangular arrays. Use bounding box over valid pixels and fill invalid from gt.
    yy, xx = np.where(valid)
    y0, y1, x0, x1 = yy.min(), yy.max()+1, xx.min(), xx.max()+1
    pp = pred[y0:y1, x0:x1].copy(); gg = gt[y0:y1, x0:x1].copy(); vv = valid[y0:y1, x0:x1]
    pp[~vv] = gg[~vv]
    ssim = float(ssim_fn(np.clip(pp,0,1), np.clip(gg,0,1), data_range=1.0))
    return dict(mse=mse, psnr=psnr, ssim=ssim)

def row_metrics(pred, base, gt, valid, tag, t0, tt, t3):
    mr = eval_metrics_single(pred, gt, valid)
    br = eval_metrics_single(base, gt, valid)
    row = {"window_start": t0, "target_time": tt, "window_end": t3, "target": tag, "valid_pixels": int(valid.sum())}
    for k, v in br.items(): row[f"baseline_{k}"] = v
    for k, v in mr.items(): row[f"model_{k}"] = v
    return row

## 10. True +10/+20 evaluation if timestamps exist

In [ ]:
time_to_idx = {t: i for i, t in enumerate(times)}
true_rows = []
for i, t0 in enumerate(times):
    t1 = t0 + dt.timedelta(minutes=10)
    t2 = t0 + dt.timedelta(minutes=20)
    t3 = t0 + dt.timedelta(minutes=30)
    if t1 in time_to_idx and t2 in time_to_idx and t3 in time_to_idx:
        j1, j2, j3 = time_to_idx[t1], time_to_idx[t2], time_to_idx[t3]
        valid = masks[i] & masks[j1] & masks[j2] & masks[j3]
        print(f"true eval window: {t0:%H:%M}, {t1:%H:%M}, {t2:%H:%M}, {t3:%H:%M}")
        p1 = interp_tiled(frames[i], frames[j3], masks[i], masks[j3], 1/3)
        b1 = (2/3)*frames[i] + (1/3)*frames[j3]
        true_rows.append(row_metrics(p1, b1, frames[j1], valid, "true_plus_10", t0, t1, t3))
        p2 = interp_tiled(frames[i], frames[j3], masks[i], masks[j3], 2/3)
        b2 = (1/3)*frames[i] + (2/3)*frames[j3]
        true_rows.append(row_metrics(p2, b2, frames[j2], valid, "true_plus_20", t0, t2, t3))

true_metrics = pd.DataFrame(true_rows)
if len(true_metrics):
    path = OUT_DIR / "true_plus10_plus20_metrics.csv"
    true_metrics.to_csv(path, index=False)
    print("saved", path)
    display(true_metrics.describe(numeric_only=True))
    display(true_metrics.head())
else:
    print("No true +10/+20 windows found. This usually means the INSAT product cadence is 30 minutes.")

## 11. Proxy evaluation when only 30-minute cadence exists

In [ ]:
proxy_rows = []
limit = (N - 3) if MAX_EVAL_WINDOWS is None else min(MAX_EVAL_WINDOWS, N - 3)
if limit <= 0:
    print("Not enough frames for proxy evaluation. Increase COUNT to at least 4.")
else:
    for i in range(limit):
        f0, y1, y2, f3 = frames[i], frames[i+1], frames[i+2], frames[i+3]
        m0, m1, m2, m3 = masks[i], masks[i+1], masks[i+2], masks[i+3]
        valid = m0 & m1 & m2 & m3
        print(f"proxy window {i+1}/{limit}: {times[i]:%H:%M} -> {times[i+3]:%H:%M}")
        p1 = interp_tiled(f0, f3, m0, m3, 1/3)
        b1 = (2/3)*f0 + (1/3)*f3
        proxy_rows.append(row_metrics(p1, b1, y1, valid, "proxy_alpha_1_3", times[i], times[i+1], times[i+3]))
        p2 = interp_tiled(f0, f3, m0, m3, 2/3)
        b2 = (1/3)*f0 + (2/3)*f3
        proxy_rows.append(row_metrics(p2, b2, y2, valid, "proxy_alpha_2_3", times[i], times[i+2], times[i+3]))

proxy_metrics = pd.DataFrame(proxy_rows)
if len(proxy_metrics):
    path = OUT_DIR / "proxy_metrics.csv"
    proxy_metrics.to_csv(path, index=False)
    print("saved", path)
    display(proxy_metrics.describe(numeric_only=True))
    display(proxy_metrics.head())

## 12. Direct +10/+20 prediction from consecutive INSAT frames

In [ ]:
direct_rows = []
full_preds = {}
for i in range(N - 1):
    t0, t3 = times[i], times[i+1]
    gap = (t3 - t0).total_seconds() / 60
    print(f"direct pair {i+1}/{N-1}: {t0:%H:%M} -> {t3:%H:%M} gap={gap:.0f} min")
    p1 = interp_tiled(frames[i], frames[i+1], masks[i], masks[i+1], 1/3)
    p2 = interp_tiled(frames[i], frames[i+1], masks[i], masks[i+1], 2/3)
    for tag, pred, alpha in [("plus_1_3_gap", p1, 1/3), ("plus_2_3_gap", p2, 2/3)]:
        tt = t0 + (t3 - t0) * alpha
        valid = masks[i] & masks[i+1]
        direct_rows.append({
            "start_time": t0, "pred_time": tt, "end_time": t3, "gap_minutes": gap,
            "target": tag, "valid_pixels": int(valid.sum()),
            "pred_norm_mean": float(pred[valid].mean()),
            "pred_bt_mean_k": float((pred[valid] * (BT_MAX-BT_MIN) + BT_MIN).mean()),
        })
        png = (np.clip(pred, 0, 1) * 255).astype(np.uint8)
        png_path = OUT_DIR / f"pred_tir1_{tt:%Y%m%d_%H%M}_{tag}.png"
        imageio.imwrite(png_path, png)
        if SAVE_FULL_NPZ:
            full_preds[f"{tt:%Y%m%d_%H%M}_{tag}"] = pred.astype(np.float16)

direct = pd.DataFrame(direct_rows)
path = OUT_DIR / "direct_prediction_summary.csv"
direct.to_csv(path, index=False)
print("saved", path)
if SAVE_FULL_NPZ:
    np.savez_compressed(OUT_DIR / "direct_predictions_float16.npz", **full_preds)
display(direct.head())

## 13. Visual QA

In [ ]:
PAIR_I = 0
p1 = interp_tiled(frames[PAIR_I], frames[PAIR_I+1], masks[PAIR_I], masks[PAIR_I+1], 1/3)
p2 = interp_tiled(frames[PAIR_I], frames[PAIR_I+1], masks[PAIR_I], masks[PAIR_I+1], 2/3)

fig, ax = plt.subplots(1, 4, figsize=(16, 4.5))
items = [
    (frames[PAIR_I], f"real {times[PAIR_I]:%H:%M}"),
    (p1, "model alpha=1/3"),
    (p2, "model alpha=2/3"),
    (frames[PAIR_I+1], f"real {times[PAIR_I+1]:%H:%M}"),
]
for a, (img, title) in zip(ax, items):
    a.imshow(img, cmap="gray_r", vmin=0, vmax=1)
    a.set_title(title)
    a.axis("off")
plt.tight_layout()
fig_path = OUT_DIR / "direct_pair_visual.png"
plt.savefig(fig_path, dpi=140, bbox_inches="tight")
print("saved", fig_path)
plt.show()

## Outputs

The notebook writes outputs to `/kaggle/working/insat3dr_tir1_advectnet_outputs`:

- `true_plus10_plus20_metrics.csv` if real 10-minute targets exist.
- `proxy_metrics.csv` for half-hourly-only sequences.
- `direct_prediction_summary.csv` for all direct predictions.
- `pred_tir1_*.png` predicted normalized TIR1 brightness-temperature maps.
- `direct_pair_visual.png` quick visual QA.